# Physics Parameter Estimation with PD-iPINN

This notebook demonstrates **physics parameter estimation** using the Coupled (PD-iPINN) model.

**Goal**: Estimate diffusion coefficient (D) or decay coefficient (k) from noisy voltage observations.

**Runtime**: ~5 minutes on Colab GPU

In [ ]:
# =============================================================================
# IMPORTANT: Set backend BEFORE importing deepxde
# =============================================================================
import os
os.environ['DDE_BACKEND'] = 'tensorflow.compat.v1'

# Install dependencies (Colab)
!pip install deepxde==1.14.0 --quiet

In [ ]:
# =============================================================================
# Imports
# =============================================================================
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

print(f"DeepXDE: {dde.__version__}")
print(f"Backend: {dde.backend.backend_name}")

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

# Random seed for reproducibility
SEED = 42

# Noise level (0.0 to 0.5)
NOISE_LEVEL = 0.30  # 30% noise | # Parameter estimation becomes unreliable above 30% noise (see paper).

# Grid resolution
NUM_X = 51   # Spatial grid points
NUM_T = 101  # Temporal grid points

# Training settings
EPOCHS = 5000        # More epochs for parameter estimation
DISPLAY_EVERY = 1000
NUM_DOMAIN = 2000

# Physics Parameters (Ground Truth)
EPS = 1.0            # Permittivity (fixed)
D_TRUE = 0.01        # True diffusion coefficient [m²/s]
K_TRUE = 0.5         # True decay coefficient [1/s]

# Parameter Estimation Settings (INDIVIDUAL CONTROL)
LEARN_D = True       # Learn diffusion coefficient?
LEARN_K = False       # Learn decay coefficient?

# Initial guesses (used when LEARN_X = True)
D_INIT = 0.05        # Initial guess for D (5x off)
K_INIT = 1.0         # Initial guess for k (2x off)

# Derived parameters (for data generation)
DECAY_RATE = D_TRUE * (np.pi ** 2) + K_TRUE

# Domain bounds
X_MIN, X_MAX = -1.0, 1.0
T_MIN, T_MAX = 0.0, 1.0

print(f"Configuration: noise={NOISE_LEVEL*100:.0f}%, grid=({NUM_X}×{NUM_T})")
print(f"\nTrue parameters:")
print(f"  D = {D_TRUE}")
print(f"  k = {K_TRUE}")
print(f"\nParameter learning:")
print(f"  LEARN_D = {LEARN_D}" + (f" (init: {D_INIT}, error: {abs(D_INIT-D_TRUE)/D_TRUE*100:.0f}%)" if LEARN_D else " (fixed)"))
print(f"  LEARN_K = {LEARN_K}" + (f" (init: {K_INIT}, error: {abs(K_INIT-K_TRUE)/K_TRUE*100:.0f}%)" if LEARN_K else " (fixed)"))

In [ ]:
# =============================================================================
# Exact Solutions
# =============================================================================

def phi_ex_func(X):
    """Exact potential: φ_ex(x,t) = sin(πx) exp(-λt)"""
    x, t = X[:, 0:1], X[:, 1:2]
    return np.sin(np.pi * x) * np.exp(-DECAY_RATE * t)

def rho_ex_func(X):
    """Exact charge density: ρ_ex(x,t) = επ² sin(πx) exp(-λt)"""
    x, t = X[:, 0:1], X[:, 1:2]
    return EPS * (np.pi ** 2) * np.sin(np.pi * x) * np.exp(-DECAY_RATE * t)

def add_noise(phi_clean, noise_level, seed):
    """Add relative Gaussian noise: φ_obs = φ_ex + σ·max|φ_ex|·η"""
    np.random.seed(seed)
    if noise_level == 0:
        return phi_clean.copy()
    phi_scale = np.max(np.abs(phi_clean))
    noise = noise_level * phi_scale * np.random.randn(*phi_clean.shape)
    return phi_clean + noise

In [ ]:
# =============================================================================
# Generate Data
# =============================================================================

# Create grid
x_grid = np.linspace(X_MIN, X_MAX, NUM_X)
t_grid = np.linspace(T_MIN, T_MAX, NUM_T)
X_mesh, T_mesh = np.meshgrid(x_grid, t_grid, indexing='ij')
XT_data = np.hstack([X_mesh.flatten()[:, None], T_mesh.flatten()[:, None]])

# Ground truth
phi_ex = phi_ex_func(XT_data)
rho_ex = rho_ex_func(XT_data)

# Add noise
phi_obs = add_noise(phi_ex, NOISE_LEVEL, SEED)

In [ ]:
# =============================================================================
# Training Function
# =============================================================================

def train_model(phi_obs, XT_data, seed):
    tf.compat.v1.reset_default_graph()
    dde.config.set_random_seed(seed)
    np.random.seed(seed)

    # Create Variables (trainable or fixed)
    if LEARN_D:
        D_var = dde.Variable(D_INIT)
    else:
        D_var = D_TRUE

    if LEARN_K:
        k_var = dde.Variable(K_INIT)
    else:
        k_var = K_TRUE

    # PDE Definition (uses D_var and k_var)
    def pde_coupled(X, y):
        phi_xx = dde.grad.hessian(y, X, component=0, i=0, j=0)
        rho = y[:, 1:2]
        rho_t = dde.grad.jacobian(y, X, i=1, j=1)
        rho_xx = dde.grad.hessian(y, X, component=1, i=0, j=0)

        res_poisson = -phi_xx - rho / EPS
        res_diffusion = rho_t - D_var * rho_xx + k_var * rho

        return [res_poisson, res_diffusion]

    # Setup Model
    geom = dde.geometry.Interval(X_MIN, X_MAX)
    timedomain = dde.geometry.TimeDomain(T_MIN, T_MAX)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)

    observe_phi = dde.icbc.PointSetBC(XT_data, phi_obs, component=0)
    loss_weights = [1, 1, 1000]

    data = dde.data.TimePDE(
        geomtime, pde_coupled, [observe_phi],
        num_domain=NUM_DOMAIN, num_boundary=0, num_initial=0,
        anchors=XT_data, num_test=5000
    )

    net = dde.nn.PFNN(
        [2, [64, 64], [64, 64], [64, 64], [64, 64], 2],
        "tanh", "Glorot uniform"
    )

    model = dde.Model(data, net)

    # Setup Callback for Parameter Tracking
    external_vars = []
    var_names = []

    if LEARN_D:
        external_vars.append(D_var)
        var_names.append('D')
    if LEARN_K:
        external_vars.append(k_var)
        var_names.append('k')

    if external_vars:
        variable_callback = dde.callbacks.VariableValue(
            external_vars,
            period=DISPLAY_EVERY,
            filename="variables.dat"
        )
        callbacks = [variable_callback]
    else:
        callbacks = []

    # Compile and Train
    model.compile("adam", lr=1e-3, loss_weights=loss_weights,
                  external_trainable_variables=external_vars if external_vars else None)

    learn_status = []
    if LEARN_D:
        learn_status.append(f"D (init={D_INIT})")
    if LEARN_K:
        learn_status.append(f"k (init={K_INIT})")
    status_str = ", ".join(learn_status) if learn_status else "None (all fixed)"

    print(f"\nTraining PD-iPINN...")
    print(f"  Learning: {status_str}")

    losshistory, _ = model.train(
        epochs=EPOCHS,
        display_every=DISPLAY_EVERY,
        callbacks=callbacks
    )

    # Get Final Parameter Values
    if LEARN_D:
        D_final = model.sess.run(D_var)
    else:
        D_final = D_TRUE

    if LEARN_K:
        k_final = model.sess.run(k_var)
    else:
        k_final = K_TRUE

    # Predict
    output = model.predict(XT_data)
    phi_pred = output[:, 0:1]
    rho_pred = output[:, 1:2]

    tf.keras.backend.clear_session()

    return phi_pred, rho_pred, D_final, k_final, var_names, losshistory

print("Training function defined")

In [ ]:
# =============================================================================
# Train Model
# =============================================================================

phi_pred, rho_pred, D_est, k_est, var_names, losshistory = train_model(phi_obs, XT_data, SEED)

In [ ]:
# =============================================================================
# Compute Metrics
# =============================================================================

def relative_L2_error(pred, exact):
    """Relative L2 error: ||pred - exact||₂ / ||exact||₂"""
    return np.sqrt(np.mean((pred - exact)**2)) / np.sqrt(np.mean(exact**2))

phi_L2 = relative_L2_error(phi_pred, phi_ex)
rho_L2 = relative_L2_error(rho_pred, rho_ex)

print("\n" + "="*60)
print(f"Results (noise = {NOISE_LEVEL*100:.0f}%)")
print("="*60)

print(f"\nReconstruction Errors:")
print(f"  φ L2 Error: {phi_L2*100:.2f}%")
print(f"  ρ L2 Error: {rho_L2*100:.2f}%")

if LEARN_D or LEARN_K:
    print(f"\n" + "-"*60)
    print(f"Parameter Estimation:")
    print(f"-"*60)
    print(f"  {'Param':<8} {'True':<12} {'Init':<12} {'Estimated':<12} {'Error':<10} {'Status'}")
    print(f"  {'-'*70}")

    D_error = abs(D_est - D_TRUE) / D_TRUE * 100
    D_status = "learned" if LEARN_D else "fixed"
    D_init_str = f"{D_INIT}" if LEARN_D else "-"
    print(f"  {'D':<8} {D_TRUE:<12.4f} {D_init_str:<12} {D_est:<12.6f} {D_error:<10.2f}% {D_status}")

    k_error = abs(k_est - K_TRUE) / K_TRUE * 100
    k_status = "learned" if LEARN_K else "fixed"
    k_init_str = f"{K_INIT}" if LEARN_K else "-"
    print(f"  {'k':<8} {K_TRUE:<12.4f} {k_init_str:<12} {k_est:<12.6f} {k_error:<10.2f}% {k_status}")

print("="*60)

In [ ]:
# =============================================================================
# Load Parameter History from Callback
# =============================================================================

param_history = {'epochs': [], 'D': [], 'k': []}

if (LEARN_D or LEARN_K) and os.path.exists("variables.dat"):
    with open("variables.dat", "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                epoch = float(parts[0])
                param_history['epochs'].append(epoch)

                col_idx = 1
                if LEARN_D:
                    D_val = float(parts[col_idx].strip('[]'))
                    param_history['D'].append(D_val)
                    col_idx += 1
                else:
                    param_history['D'].append(D_TRUE)

                if LEARN_K:
                    k_val = float(parts[col_idx].strip('[]'))
                    param_history['k'].append(k_val)
                else:
                    param_history['k'].append(K_TRUE)

    print(f"Loaded parameter history: {len(param_history['epochs'])} records")
else:
    print("No parameter history (all parameters fixed or file not found)")

In [ ]:
# =============================================================================
# Setup for plotting
# =============================================================================
def reshape_to_grid(arr):
    return arr.reshape(NUM_X, NUM_T)

phi_ex_grid = reshape_to_grid(phi_ex)
rho_ex_grid = reshape_to_grid(rho_ex)
phi_obs_grid = reshape_to_grid(phi_obs)
phi_pred_grid = reshape_to_grid(phi_pred)
rho_pred_grid = reshape_to_grid(rho_pred)

import matplotlib as mpl
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['STIXGeneral'],
    'mathtext.fontset': 'stix',
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'axes.linewidth': 0.8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.fontsize': 7,
    'legend.frameon': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

DOUBLE_COL = 6.69
C_TRUE, C_EST = '#000000', '#0072B2'
t_indices = [0, 25, 50, 75, 100]
t_values = [0, 0.25, 0.5, 0.75, 1.0]

In [ ]:
# =============================================================================
# Figure 1
# =============================================================================
if (LEARN_D or LEARN_K) and len(param_history['epochs']) > 1:
    n_param_plots = int(LEARN_D) + int(LEARN_K)
    n_plots = n_param_plots * 2  # 각 파라미터마다 값 + 오차
    fig, axes = plt.subplots(1, n_plots, figsize=(DOUBLE_COL, DOUBLE_COL * 0.3))

    plot_idx = 0

    if LEARN_D:
        # D convergence
        ax = axes[plot_idx]
        ax.axhline(y=D_TRUE, color='k', linestyle='--', lw=0.8, label=f'True ({D_TRUE})')
        ax.plot(param_history['epochs'], param_history['D'], '-', color=C_EST, lw=1.0, label='Estimated')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('D')
        ax.set_title(f'(a) D convergence')
        ax.legend(loc='best', fontsize=6)
        ax.set_xlim(0, EPOCHS)
        plot_idx += 1

        # D relative error
        ax = axes[plot_idx]
        D_error = [abs(d - D_TRUE) / D_TRUE * 100 for d in param_history['D']]
        ax.plot(param_history['epochs'], D_error, '-', color=C_EST, lw=1.0)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Relative Error (%)')
        ax.set_title(f'(b) D error: {D_error[-1]:.1f}%')
        ax.set_xlim(0, EPOCHS)
        ax.set_ylim(0, max(D_error) * 1.1)
        plot_idx += 1

    if LEARN_K:
        # k convergence
        ax = axes[plot_idx]
        ax.axhline(y=K_TRUE, color='k', linestyle='--', lw=0.8, label=f'True ({K_TRUE})')
        ax.plot(param_history['epochs'], param_history['k'], '-', color=C_EST, lw=1.0, label='Estimated')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('k')
        label_a = '(c)' if LEARN_D else '(a)'
        ax.set_title(f'{label_a} k convergence')
        ax.legend(loc='best', fontsize=6)
        ax.set_xlim(0, EPOCHS)
        plot_idx += 1

        # k relative error
        ax = axes[plot_idx]
        k_error = [abs(k - K_TRUE) / K_TRUE * 100 for k in param_history['k']]
        ax.plot(param_history['epochs'], k_error, '-', color=C_EST, lw=1.0)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Relative Error (%)')
        label_b = '(d)' if LEARN_D else '(b)'
        ax.set_title(f'{label_b} k error: {k_error[-1]:.1f}%')
        ax.set_xlim(0, EPOCHS)
        ax.set_ylim(0, max(k_error) * 1.1)

    fig.tight_layout()
    plt.savefig('param_convergence.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Parameter convergence plot skipped (no parameters being learned)")

In [ ]:
# =============================================================================
# Figure 2
# =============================================================================
fig, axes = plt.subplots(2, 5, figsize=(DOUBLE_COL, DOUBLE_COL * 0.42))

for i, (t_idx, t_val) in enumerate(zip(t_indices, t_values)):
    # Top: φ
    ax = axes[0, i]
    ax.plot(x_grid, phi_ex_grid[:, t_idx], '-', color=C_TRUE, lw=0.8, label='True')
    ax.plot(x_grid, phi_pred_grid[:, t_idx], '-', color=C_EST, lw=0.8, label='PD-iPINN')
    ax.set_title(f'$t = {t_val}$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xlabel(r'$x$')
    if i == 0:
        ax.set_ylabel(r'$\phi$')
        ax.legend(loc='lower right', fontsize=4)

    # Bottom: ρ
    ax = axes[1, i]
    ax.plot(x_grid, rho_ex_grid[:, t_idx], '-', color=C_TRUE, lw=0.8)
    ax.plot(x_grid, rho_pred_grid[:, t_idx], '-', color=C_EST, lw=0.8)
    ax.set_xlabel(r'$x$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-11, 11)
    if i == 0:
        ax.set_ylabel(r'$\rho$')

fig.tight_layout()
fig.text(0.5, -0.02, rf'Noise: $\sigma = {int(NOISE_LEVEL*100)}\%$, D={D_est:.4f}, k={k_est:.4f}',
         ha='center', va='top', fontsize=7)
plt.savefig('demo_param_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

This demo shows that **PD-iPINN** can simultaneously:

1. **Reconstruct** the charge density ρ from noisy voltage observations
2. **Estimate** physics parameters (D or k) from data

---

### Important: Identifiability

Simultaneous estimation of both D and k is **not identifiable** from voltage observations alone. Estimate one parameter at a time:

| Setting | Use case |
|---------|----------|
| `LEARN_D=True, LEARN_K=False` | Estimate D (knowing k) |
| `LEARN_D=False, LEARN_K=True` | Estimate k (knowing D) |

Setting both `True` may lead to non-unique solutions due to parameter coupling.

---

### Model Naming Convention

| Code | Paper | Description |
|------|-------|-------------|
| `Coupled` | PD-iPINN | Poisson + Diffusion-Decay |